# Function 7 · Week 10 decision review

Six-input hyperparameter tuning on two regions

[Detailed observations](observation_review.ipynb) · [Detailed fits](model_predictions.ipynb) · [Previous pooled study](full-cohort/model_predictions.ipynb) · [2024 comparison](../../../side-quests/historical-cohort-2024/README.md) · [Course data-use review](../../../docs/capstone-data-policy.md)

**Decision status.** Week 10 is prepared and unsubmitted. Round 9 is the latest official return. Start with the [prepared comparison](#prepared-week-10-decision) when choosing between the two portal batches.

## Expectations

Round 9 used the kernel-ridge alternative near earlier good points, with x1 fixed at zero. The saved weekly GP and neural models provide a common pre-return forecast.

In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys

from IPython.display import display
import numpy as np
import pandas as pd

FUNCTION = 7
WEEK_DIR = Path.cwd().resolve()
WEEK = int(WEEK_DIR.name.removeprefix("Week_"))
STAGE = WEEK_DIR.parents[2]
SHARED = STAGE / "notebooks/Shared"
SPECIAL = STAGE / "notebooks/Function_1/FunctionSpecificShared"
sys.path[:0] = [str(SHARED), str(SPECIAL)]
from weekly_peak_review import peak_review, portal_point
from query_budget import remaining_queries

analysis = WEEK_DIR / "official" if FUNCTION <= 4 else WEEK_DIR
official_path = analysis / "observations.csv"
model_snapshot = json.loads((analysis / "model-review.json").read_bytes())
observation_snapshot = json.loads((analysis / "observation-review.json").read_bytes())
feedback = json.loads((STAGE / f"submissions/round-{WEEK-1:02d}-results.json").read_bytes())
record = feedback["results"][FUNCTION - 1]
forecast = observation_snapshot["historical_forecast"]
if (model_snapshot["week"], model_snapshot["function"], model_snapshot["snapshot_sha256"]) != (
        WEEK, FUNCTION, sha256(official_path.read_bytes()).hexdigest()):
    raise ValueError("The official model review and observations describe different snapshots")
if (feedback["round"], record["function"], record["portal_input"], record["output"]) != (
        WEEK - 1, FUNCTION, forecast["portal_input"], forecast["actual_output"]):
    raise ValueError("The saved forecast must match the returned observation")
official = pd.read_csv(official_path, float_precision="round_trip")
cols = [c for c in official if c.startswith("x")]
x = official[cols].to_numpy(float)
raw_y = official.y.to_numpy(float)
later = official.source_round.to_numpy() > 0
best_row = official.loc[official.y.idxmax()]
incumbent = best_row[cols].to_numpy(float)
queries_left = remaining_queries(WEEK_DIR)
units = forecast["prediction_units"]
target = raw_y
if FUNCTION == 1:
    from function_1_model_helpers import signed_log
    target = signed_log(raw_y, scale=forecast["height_scale"])

display(pd.Series({
    "Archived forecast source": forecast["source"],
    "Submitted input": forecast["portal_input"],
    "Forecast units": units,
    "Saved GP mean": forecast["gp_mean"],
    "GP latent SD": forecast["gp_latent_sd"],
    "Saved neural mean": forecast["neural_mean"],
    "Neural seed SD": forecast["neural_seed_sd"],
}, name="Forecast frozen before the return"))

Archived forecast source    notebooks/Function_7/Week_09/figures/neural-ne...
Submitted input             0.000000-0.145572-0.392685-0.242554-0.346193-0...
Forecast units                                                       output y
Saved GP mean                                                        2.643818
GP latent SD                                                         0.091065
Saved neural mean                                                     2.72038
Neural seed SD                                                       0.021499
Name: Forecast frozen before the return, dtype: object

## Results

The new point at x1 = 0 scored below the best. The 2024 and independent campaign records above 3.3 support a region with positive x1. The boundary and interior hypotheses are both worth distinguishing.

In [2]:
latest_fitted = forecast["actual_in_prediction_units"]
display(pd.Series({"Returned score": record["output"], "Best before return": record["best_before"],
                   "Best now": record["best_after"], "Return rank": observation_snapshot["latest_rank"],
                   "Actual in forecast units": latest_fitted,
                   "Actual minus GP mean": forecast["actual_minus_gp_mean"],
                   "Actual minus neural mean": forecast["actual_minus_neural_mean"]}, name="Official result"))

# These references describe observed outcomes. The decision fits use official rows.
def optional_reference(folder, file, audit_name, expected):
    path = folder / file
    manifest = folder / audit_name
    if not (path.is_file() and manifest.is_file()):
        return None
    manifest_data = json.loads(manifest.read_bytes())
    if sha256(path.read_bytes()).hexdigest() != expected(manifest_data):
        raise ValueError(f"Reference hash mismatch: {path}")
    return pd.read_csv(path, float_precision="round_trip")

current_reference = optional_reference(
    STAGE / "data/cohort/2026-11-25", f"function_{FUNCTION}/observations.csv", "dataset-audit.json",
    lambda a: a["summary"][FUNCTION - 1]["sha256"],
)
archive = optional_reference(
    STAGE / "data/historical-cohort/2024", f"function_{FUNCTION}.csv", "intake-audit.json",
    lambda a: a["functions"][str(FUNCTION)]["sha256"],
)
reference_rows = []
if current_reference is not None:
    peer = current_reference.loc[current_reference.observation_id.str.startswith("peer-")]
    reference_rows.append(("Published current cohort", peer))
if archive is not None:
    reference_rows.append(("2024 archive, including extra starters", archive))
benchmarks = []
for label, frame in reference_rows:
    row = frame.loc[frame.y.idxmax()]
    benchmarks.append({"Scope": label, "Highest measured": row.y,
                       "Difference from your best": row.y - best_row.y,
                       "Distance to your best": np.linalg.norm(row[cols].to_numpy(float) - incumbent)})
if benchmarks:
    display(pd.DataFrame(benchmarks).round(8))
else:
    print("Local comparison snapshots are unavailable for this run.")

Returned score              2.548888
Best before return          2.739375
Best now                    2.739375
Return rank                 5.000000
Actual in forecast units    2.548888
Actual minus GP mean       -0.094930
Actual minus neural mean   -0.171492
Name: Official result, dtype: float64

,Scope,Highest measured,Difference from your best,Distance to your best
0,Published current cohort,2.688768,-0.050608,0.263878
1,"2024 archive, including extra starters",3.319276,0.579901,0.239884


In [3]:
def by_x1(frame):
    return frame.assign(region=np.where(frame["x1"].round(6).eq(0), "x1 = 0", "x1 > 0")) \
                .groupby("region")["y"].agg(["count", "median", "max"])

display(by_x1(official).round(8))
if archive is not None:
    display(archive.nlargest(5, "y")[list(cols) + ["y"]].round(8))

,count,median,max
region,,,
x1 = 0,5,2.548888,2.739375
x1 > 0,34,0.094028,2.618968


,x1,x2,x3,x4,x5,x6,y
42,0.215055,0.156395,0.470335,0.277514,0.312339,0.658909,3.319276
93,0.217215,0.157590,0.452435,0.276118,0.307239,0.659013,3.313261
92,0.215847,0.159363,0.447721,0.276331,0.307725,0.662010,3.310996
94,0.225971,0.162570,0.458913,0.281655,0.307708,0.661830,3.309524
81,0.220000,0.166667,0.450000,0.273333,0.306667,0.653333,3.307472


## Predictions

**Local peak and global evidence**

Your measurements support a high-score boundary region. Published measured values beat it elsewhere, so the observed point has no defensible global claim. A single face-only search cannot adjudicate the two regions.

### Prepared Week 10 decision

**Prepared and unsubmitted as of 25 November 2026.** Round 9 is the latest confirmed feedback. Four portal queries remain per function, including Week 10. Use the cohort-informed route only if the course permits that use, including the exact published F2 repeat. The official-data fallback is ready if the rules remain unclear near the portal deadline.

The revised input is the coordinatewise median of six distinct archival points above 3.3 and one independent corroboration. It shifts x1 and x3 into the stronger measured region. The fallback changes x3 on the official x1=0 face. The owner GP retains that face because the broader interior basin lacks nearby official coverage.

**Week 11 depends on the return.** Use a competitive interior return to justify a local follow-up. Otherwise use the face as a control and reassess with the remaining budget.

[Week 10 course check and exact batches](../../../submissions/Week_10/README.md) · [Cohort advice and replay](../../../submissions/Week_10/advisory-review.md)

The estimates below use the advisory review's official-data GP comparator and predictive return standard deviation, which includes modelled observation noise. F8 uses the validated starter quadratic and omits a standard deviation because that fit supplies no probabilistic interval. The rows are proposed inputs with model estimates.

In [4]:
from week_10_decision_record import decision_comparison

with pd.option_context("display.max_colwidth", None):
    display(decision_comparison(STAGE, FUNCTION, official))

,Portal input,Official GP mean (score),Predictive return SD
Stage,,,
Initial proposal,0.000000-0.135108-0.370684-0.306265-0.311144-0.700000,2.720164,0.035610
Advisory experiment,0.000000-0.135108-0.565684-0.306265-0.311144-0.649736,2.735340,0.082141
Revised cohort-informed,0.216539-0.157590-0.452435-0.276118-0.307725-0.659013,2.288924,0.201802
Official-data fallback,0.000000-0.135108-0.498084-0.306265-0.311144-0.649736,2.746535,0.055247


### Conditional peak check

Two Gaussian processes with fitted noise share the same candidate set. A local candidate lies within the stated Euclidean radius of the observed incumbent. The columns report the fraction of joint latent draws in which none of the candidates in that set beats the incumbent by more than the working tolerance. The broader search includes Sobol points, corners, observed coordinates and surrogate-suggested probes. It cannot certify the continuous function or rule out unrepresented peaks. The nine grouped official return holdouts and nominal 90% observation-band counts indicate predictive fit. They do not calibrate the global-maximum event. The tolerance and radius are analysis choices.

In [5]:
radius = 0.13
tolerance = 0.08
peak = peak_review(x, target, later, radius=radius, tolerance=tolerance,
                   length_scales=[0.19, 0.22, 0.26, 0.23, 0.24, 0.29], noise=[0.018, 1e-05, 0.35],
                   minimum_distance=model_snapshot["minimum_query_distance"])
print(f"Local radius {radius:g}. Score tolerance {tolerance:g} in {units}.")
print(f"Joint posterior over {peak['candidate_count']} candidates, including {peak['local_count']} local."
      f" {peak['probability_draws']} draws per kernel.")
display(peak["summary"].round(5))
print("Adaptive two-query comparison on the Matérn shortlist")
one_query = peak["acquisitions"]["Matérn 3/2"].nlargest(1, "One-query EI")
print(f"Best one-query input: {one_query.iloc[0]['Input']}")
display(peak["acquisitions"]["Matérn 3/2"].head(5).round(6))

Local radius 0.13. Score tolerance 0.08 in output y.
Joint posterior over 510 candidates, including 153 local. 1024 draws per kernel.


,Local set within tolerance,Broader set within tolerance,Broader gap median,Broader gap 90th pct,Return MAE,Return RMSE,In 90% observation band,Return holdouts,Fitted noise SD,Fit warnings
Model,,,,,,,,,,
Matérn 3/2,0.64941,0.64453,0.06388,0.13710,0.12332,0.15907,5,9,0.00305,2
RBF,0.49512,0.45215,0.08444,0.15537,0.15782,0.17581,5,9,0.08636,1


Adaptive two-query comparison on the Matérn shortlist
Best one-query input: 0.000000-0.135108-0.498084-0.306265-0.311144-0.649736


,Input,Region,Mean,Observed SD,One-query EI,Two-query gain
0,0.000000-0.135108-0.498084-0.306265-0.311144-0...,Local,2.746535,0.055247,0.025805,0.037188
1,0.000000-0.126852-0.480475-0.324867-0.321732-0...,Local,2.738525,0.052605,0.020564,0.036708
4,0.000000-0.167608-0.370684-0.306265-0.311144-0...,Local,2.747760,0.026723,0.015374,0.036101
2,0.000000-0.190452-0.449361-0.280723-0.297085-0...,Local,2.719535,0.062177,0.016137,0.035718
3,0.000000-0.153709-0.438789-0.285586-0.314010-0...,Local,2.731387,0.049014,0.015819,0.034316


### Strategy for the remaining queries

Use two small trust regions and screen coordinates through held-out fit quality before reducing dimension. TuRBO motivates moving the search budget between regions. Random embeddings are a published alternative when effective dimension is low, an assumption that needs evidence here.

The code compares one-query expected improvement in the returned score with a two-query approximation. It conditions on a first outcome, then chooses a different point on a shortlist. Both steps apply the saved minimum-spacing rule. It holds fitted kernel parameters fixed. Refit after each actual result and reassess the next action. The four rounds below are conditional decisions.

| Round | Action given the evidence so far |
| --- | --- |
| 10 | The revised route tests the archive x1/x3 basin. The official fallback changes x3 on the x1=0 face. |
| 11 | Use a competitive interior return to justify a local follow-up. Otherwise use the face as a control and reassess with the remaining budget. |
| 12 | Spend the third query in the region with higher measured upside, using a smaller trust region after a successful move. |
| 13 | Refine the best confirmed region with the final query. |

[Eriksson et al., local Bayesian optimisation](https://papers.nips.cc/paper_files/paper/2019/hash/6c990b7aca7bc7058f5e98ea909e924b-Abstract.html) · [Wang et al., random embeddings](https://ijcai.org/proceedings/2013/263)

In [6]:
print(f"{queries_left} query opportunities per function, including Week {WEEK}.")
destination = WEEK_DIR / "weekly-peak-review.json"
summary = {"function": FUNCTION, "week": WEEK, "queries_remaining": queries_left,
           "official_snapshot_sha256": model_snapshot["snapshot_sha256"],
           "candidate_count": peak["candidate_count"], "local_count": peak["local_count"],
           "probability_draws": peak["probability_draws"], "local_radius": radius,
           "tolerance": tolerance, "target_units": units,
           "models": peak["summary"].reset_index().to_dict(orient="records"),
           "matérn_shortlist": peak["acquisitions"]["Matérn 3/2"].to_dict(orient="records")}
if FUNCTION == 8:
    summary["structural_fit"] = quadratic
destination.write_bytes((json.dumps(summary, indent=2, allow_nan=False) + "\n").encode())

4 query opportunities per function, including Week 10.


4029

### Course lifecycle

1. **Define the goal and scope** – choose the next official query while assessing how close the incumbent might be to the peak.
2. **Gather the data** – preserve the official snapshot, the frozen predictions and labelled published comparisons.
3. **Explore the data** – compare the returned result with local and external observations.
4. **Clean and preprocess the data** – verify hashes and portal alignment. Preserve the recorded raw score.
5. **Select and engineer features** – use the original coordinates, with a signed-log target for F1. Structural fits declare their terms separately.
6. **Define the machine-learning task** – regression supports maximisation and assessment of the incumbent.
7. **Partition the data** – withhold each official return and any matching portal coordinate for the predictive check.
8. **Select and train candidate methods** – compare two kernel assumptions and function-specific hypotheses.
9. **Evaluate and interpret the results** – judge forecast reliability and the finite-domain comparison before choosing how to spend the remaining budget.
10. **Deploy and iterate** – deployment is not applicable. The next portal return will update the review and the query plan.